# AFTERIMAGE · FIRMS Country Yearly (US + Canada)
Télécharge les CSV NASA FIRMS (Standard Processing) par capteur, année et pays,
les garde intacts dans `raw/`, crée une copie typée en Parquet dans `clean/`,
met 2020-2021 à part pour P5, et écrit un manifeste avec sommes de contrôle.

**Avant de lancer :** sur https://firms.modaps.eosdis.nasa.gov/country/ fais
clic droit sur un fichier → « Copier l'adresse du lien », et vérifie que
`URL_TEMPLATE` ci-dessous produit exactement ce format.

## 1. Configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, hashlib, time, datetime as dt
from pathlib import Path
import requests
import pandas as pd

BASE = Path('/content/drive/MyDrive/AFTERIMAGE/firms')

# Format du lien d'un fichier (à vérifier avec le lien copié) :
URL_TEMPLATE = 'https://firms.modaps.eosdis.nasa.gov/data/country/{folder}/{year}/{prefix}_{year}_{country}.csv'

# prefix = début du nom de fichier ; folder = dossier dans l'URL (à vérifier)
SENSORS = {
    'modis':      {'prefix': 'modis',      'folder': 'modis',      'years': range(2002, 2026)},
    'viirs-snpp': {'prefix': 'viirs-snpp', 'folder': 'viirs-snpp', 'years': range(2012, 2026)},
}
COUNTRIES = ['United_States', 'Canada']
SEALED_YEARS = {2020, 2021}          # réservées à P5 (test en aveugle)

RAW, CLEAN, SEALED = BASE / 'raw', BASE / 'clean', BASE / 'clean_sealed_P5'
MANIFEST = BASE / 'manifest'
for d in (RAW, CLEAN, SEALED, MANIFEST):
    d.mkdir(parents=True, exist_ok=True)


Mounted at /content/drive


## 2. Téléchargement (CSV d'origine, jamais modifiés)

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

def download(url, dest, tries=3):
    for i in range(tries):
        try:
            with requests.get(url, stream=True, timeout=120) as r:
                if r.status_code == 404:
                    return 'missing'
                r.raise_for_status()
                tmp = dest.with_suffix('.part')
                with open(tmp, 'wb') as f:
                    for chunk in r.iter_content(1 << 20):
                        f.write(chunk)
                tmp.rename(dest)
                return 'ok'
        except requests.RequestException as e:
            print(f'  essai {i+1} échoué : {e}')
            time.sleep(5 * (i + 1))
    return 'failed'

records = []
for sensor, cfg in SENSORS.items():
    for year in cfg['years']:
        for country in COUNTRIES:
            name = f"{cfg['prefix']}_{year}_{country}.csv"
            url = URL_TEMPLATE.format(folder=cfg['folder'], prefix=cfg['prefix'],
                                      year=year, country=country)
            dest = RAW / sensor / str(year) / name
            dest.parent.mkdir(parents=True, exist_ok=True)
            status = 'cached' if dest.exists() else download(url, dest)
            rec = {'sensor': sensor, 'year': year, 'country': country, 'file': name,
                   'url': url, 'status': status,
                   'retrieved_utc': dt.datetime.now(dt.timezone.utc).isoformat(timespec='seconds')}
            if dest.exists():
                rec.update(bytes=dest.stat().st_size, sha256=sha256(dest))
            records.append(rec)
            print(f'{status:8} {name}')

dl = pd.DataFrame(records)
dl.to_csv(MANIFEST / 'download_manifest.csv', index=False)
print(dl['status'].value_counts())


ok       modis_2002_United_States.csv
ok       modis_2002_Canada.csv
ok       modis_2003_United_States.csv
ok       modis_2003_Canada.csv
ok       modis_2004_United_States.csv
ok       modis_2004_Canada.csv
ok       modis_2005_United_States.csv
ok       modis_2005_Canada.csv
ok       modis_2006_United_States.csv
ok       modis_2006_Canada.csv
ok       modis_2007_United_States.csv
ok       modis_2007_Canada.csv
ok       modis_2008_United_States.csv
ok       modis_2008_Canada.csv
ok       modis_2009_United_States.csv
ok       modis_2009_Canada.csv
ok       modis_2010_United_States.csv
ok       modis_2010_Canada.csv
ok       modis_2011_United_States.csv
ok       modis_2011_Canada.csv
ok       modis_2012_United_States.csv
ok       modis_2012_Canada.csv
ok       modis_2013_United_States.csv
ok       modis_2013_Canada.csv
ok       modis_2014_United_States.csv
ok       modis_2014_Canada.csv
ok       modis_2015_United_States.csv
ok       modis_2015_Canada.csv
ok       modis_2016_United_States.

## 3. Nettoyage : types fixés, noms de colonnes communs, rien n'est supprimé

In [ ]:
COMMON = {  # nom d'origine -> nom commun
    'brightness': 'bright_main', 'bright_ti4': 'bright_main',
    'bright_t31': 'bright_aux',  'bright_ti5': 'bright_aux',
}

def conf_class(series, sensor):
    """Classe de confiance commune : low / nominal / high (règle documentée)."""
    if sensor == 'modis':  # MODIS : 0-100 ; guide C6.1 : <30 low, 30-79 nominal, >=80 high
        c = pd.to_numeric(series, errors='coerce')
        return pd.cut(c, [-1, 29, 79, 100], labels=['low', 'nominal', 'high']).astype('string')
    m = {'l': 'low', 'n': 'nominal', 'h': 'high'}  # VIIRS : l / n / h
    return series.astype('string').str.strip().str.lower().map(m).astype('string')

def clean_file(path, sensor, year, country):
    df = pd.read_csv(path, dtype={'acq_time': str, 'confidence': str, 'version': str,
                                  'satellite': str, 'instrument': str, 'daynight': str})
    n_raw = len(df)
    df = df.rename(columns=COMMON)
    df['acq_time'] = df['acq_time'].str.zfill(4)
    df['acq_datetime_utc'] = pd.to_datetime(df['acq_date'] + ' ' + df['acq_time'],
                                            format='%Y-%m-%d %H%M', utc=True)
    df['acq_date'] = pd.to_datetime(df['acq_date']).dt.date
    df['confidence_raw'] = df.pop('confidence')
    df['confidence_class'] = conf_class(df['confidence_raw'], sensor)
    df['sensor'], df['country'], df['file_year'] = sensor, country, year
    df['source_file'] = path.name
    df['is_vegetation_fire'] = df['type'].eq(0) if 'type' in df else pd.NA
    df['is_reference_platform'] = df['satellite'].isin(['A', 'Aqua'] if sensor == 'modis' else ['N'])
    assert len(df) == n_raw, 'aucune ligne ne doit être perdue au nettoyage'
    return df

def check(df, year):
    """Contrôles simples ; renvoie la liste des problèmes trouvés."""
    issues = []
    if df.empty:
        return ['fichier vide']
    years = pd.to_datetime(df['acq_date']).dt.year
    if not years.eq(year).all():
        issues.append(f'{(~years.eq(year)).sum()} lignes hors de l\'année {year}')
    if not df['latitude'].between(-90, 90).all() or not df['longitude'].between(-180, 180).all():
        issues.append('coordonnées hors limites')
    if df['acq_datetime_utc'].isna().any():
        issues.append(f"{df['acq_datetime_utc'].isna().sum()} heures illisibles")
    if df['confidence_class'].isna().any():
        issues.append(f"{df['confidence_class'].isna().sum()} confiances non reconnues")
    dup = df.duplicated(['latitude', 'longitude', 'acq_datetime_utc', 'satellite']).sum()
    if dup:
        issues.append(f'{dup} doublons exacts')
    return issues


## 4. Lancer le nettoyage + manifeste final

In [ ]:
out = []
for r in records:
    if r['status'] not in ('ok', 'cached'):
        continue
    src = RAW / r['sensor'] / str(r['year']) / r['file']
    df = clean_file(src, r['sensor'], r['year'], r['country'])
    target_dir = SEALED if r['year'] in SEALED_YEARS else CLEAN
    dest = target_dir / r['sensor'] / str(r['year']) / r['file'].replace('.csv', '.parquet')
    dest.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(dest, index=False)
    issues = check(df, r['year'])
    out.append({**r, 'rows': len(df),
                'aqua_or_snpp_rows': int(df['is_reference_platform'].sum()),
                'vegetation_rows': int(df['is_vegetation_fire'].sum()) if 'type' in df else None,
                'versions': sorted(df['version'].dropna().unique().tolist()),
                'clean_path': str(dest.relative_to(BASE)), 'clean_sha256': sha256(dest),
                'issues': issues})
    print(f"{r['file']:45} {len(df):>9} lignes  {'OK' if not issues else issues}")

manifest = {
    'dataset': 'NASA FIRMS Country Yearly Summary (Standard Processing)',
    'source_page': 'https://firms.modaps.eosdis.nasa.gov/country/',
    'countries': COUNTRIES, 'sealed_years_for_P5': sorted(SEALED_YEARS),
    'created_utc': dt.datetime.now(dt.timezone.utc).isoformat(timespec='seconds'),
    'notes': ['CSV d\'origine intacts dans raw/', 'clean/ = copie typée, aucune ligne supprimée',
              'Filtrer à l\'analyse : is_vegetation_fire et is_reference_platform',
              'Points de détection uniquement : pas de O (observabilité), pas de nuages'],
    'files': out,
}
(MANIFEST / 'manifest.json').write_text(json.dumps(manifest, indent=2, default=str))
print('Manifeste écrit :', MANIFEST / 'manifest.json')


modis_2002_United_States.csv                      79715 lignes  OK
modis_2002_Canada.csv                             29050 lignes  OK
modis_2003_United_States.csv                     114471 lignes  OK
modis_2003_Canada.csv                             52083 lignes  OK
modis_2004_United_States.csv                     158385 lignes  OK
modis_2004_Canada.csv                             76092 lignes  OK
modis_2005_United_States.csv                     171160 lignes  OK
modis_2005_Canada.csv                             33042 lignes  OK
modis_2006_United_States.csv                     126737 lignes  OK
modis_2006_Canada.csv                             36040 lignes  ['1 doublons exacts']
modis_2007_United_States.csv                     142420 lignes  OK
modis_2007_Canada.csv                             34154 lignes  OK
modis_2008_United_States.csv                     119797 lignes  OK
modis_2008_Canada.csv                             32748 lignes  OK
modis_2009_United_States.csv               